# Loss Sensitivities

Companion notebook for Tutorial 11. Compute marginal loss factors (MLFs) —
the sensitivity of total system losses to a 1 MW injection at each bus.

Case30 has nonzero branch resistance on every branch, giving meaningful
loss structure.

## AC Marginal Loss Factors

One Jacobian-transpose solve — approximately the cost of a single
Newton-Raphson iteration.

In [ ]:
import surge

net = surge.case30()

lsf = surge.losses.compute_loss_factors(net)

print(f"Total base-case losses: {lsf.base_losses_mw:.2f} MW")
print(f"Number of buses: {len(lsf.bus_numbers)}")
print()

df = lsf.to_dataframe()
print("5 highest-loss buses:")
print(df.nlargest(5, "lsf"))
print()
print("5 lowest-loss buses:")
print(df.nsmallest(5, "lsf"))

## Verify Against Finite Difference

Add a small generator at one bus and compare the observed loss change
to the analytical MLF. AC losses are nonlinear (∝ I²), so the FD secant
won't match the exact tangent — this is why the analytical method exists.

In [ ]:
ac_base = surge.solve_ac_pf(net, surge.AcPfOptions())
base_losses = sum(ac_base.p_inject_mw)

delta_mw = 0.1
perturbed = surge.case30()
perturbed.add_generator(bus=30, p_mw=delta_mw, pmax_mw=delta_mw + 1, pmin_mw=0.0, machine_id="FD")
ac_pert = surge.solve_ac_pf(perturbed, surge.AcPfOptions())
pert_losses = sum(ac_pert.p_inject_mw)

fd_mlf = (pert_losses - base_losses) / delta_mw
bus_col = lsf.bus_numbers.index(30)
analytical_mlf = lsf.lsf[bus_col]

print(f"Finite-difference MLF at bus 30: {fd_mlf:+.6f}")
print(f"Analytical MLF at bus 30:        {analytical_mlf:+.6f}")

## Stressed Operating Point

Pass a pre-computed AC solution to skip the internal solve.

In [ ]:
stressed = surge.case30()
stressed.scale_loads(1.10)
ac_stressed = surge.solve_ac_pf(stressed, surge.AcPfOptions(enforce_q_limits=True))

lsf_stressed = surge.losses.compute_loss_factors(stressed, solution=ac_stressed)

print(f"{'':20s} {'Base':>10s} {'Stressed':>10s}")
print(f"{'Losses (MW)':20s} {lsf.base_losses_mw:10.2f} {lsf_stressed.base_losses_mw:10.2f}")
print(f"{'Max MLF':20s} {max(lsf.lsf):10.6f} {max(lsf_stressed.lsf):10.6f}")
print(f"{'Min MLF':20s} {min(lsf.lsf):10.6f} {min(lsf_stressed.lsf):10.6f}")

## Loss-Aware DC-OPF

DC power flow is lossless by definition. The penalty-factor model adjusts
each generator's effective cost so the optimizer favors low-loss locations.
The primary output is the nonzero loss component in the LMP decomposition.

In [ ]:
from surge.opf import DcOpfOptions, DcLossModel, DcOpfRuntime

net = surge.case30()

dc_lossless = surge.solve_dc_opf(
    net,
    options=DcOpfOptions(
        enforce_thermal_limits=True,
        loss_model=DcLossModel.IGNORE,
    ),
    runtime=DcOpfRuntime(lp_solver="highs"),
)

dc_lossy = surge.solve_dc_opf(
    net,
    options=DcOpfOptions(
        enforce_thermal_limits=True,
        loss_model=DcLossModel.ITERATIVE,
        loss_iterations=3,
        loss_tolerance=1e-3,
    ),
    runtime=DcOpfRuntime(lp_solver="highs"),
)

print(f"{'':25s} {'Lossless':>12s} {'Loss-aware':>12s}")
print(f"{'Total cost ($/hr)':25s} {dc_lossless.total_cost:12.2f} {dc_lossy.total_cost:12.2f}")
print()
print(f"{'Gen bus':>10s} {'Lossless':>12s} {'Loss-aware':>12s}")
for gl, ga in zip(dc_lossless.generators, dc_lossy.generators):
    print(f"{gl.bus:10d} {gl.p_mw:12.2f} {ga.p_mw:12.2f}")

## LMP Decomposition

The loss component of each bus's LMP reflects how its network position
contributes to system losses.

In [ ]:
print(f"{'Bus':>5s} {'LMP':>10s} {'Energy':>10s} {'Congestion':>10s} {'Loss':>10s}")
for b in dc_lossy.buses[:10]:
    print(f"{b.number:5d} {b.lmp:10.4f} {b.lmp_energy:10.4f} "
          f"{b.lmp_congestion:10.4f} {b.lmp_loss:10.4f}")

print()
print(f"Max |loss component|: {max(abs(b.lmp_loss) for b in dc_lossy.buses):.4f} $/MWh")
print(f"Max |cong component|: {max(abs(b.lmp_congestion) for b in dc_lossy.buses):.4f} $/MWh")